# **Data Cleaning**

## Objectives

* Apply data validation and cleaning to remove invalid records and prevent data leakage

## Inputs

* outputs/datasets/collection/HotelBookings.csv

## Outputs

* Generate cleaned dataset for analysis saved as outputs/datasets/cleaned/HotelBookingsClean.csv
* Generate a valid dataset for the cleaning and engineering pipeline to correct saved as outputs/datasets/cleaned/HotelBookingsValid.csv

## Decisions from Revenue Manager

* Duplicate bookings should remain in place
* Bookings with no guests (adults, children or babies) should be removed
* Values > 4 for babies or children and > 2 for car park spaces should be replaced with their column's median value
* Bookings with >= 5 adults should be removed
* Isolated high and low outliers for adr should be removed
* Missing data in children should be replaced by 0
* Missing data in agent and company should be replaced by 0
* Missing data in country should be replaced by the mode value 

## Standard data cleaning

* Drop variables: `['reservation_status', 'reservation_status_date', 'assigned_room_type', 'booking_changes']`
* Perform dtype correction on children, agent and company (float -> int)
* Ensure all numeric-as-categorical features (except target) are converted to 'category'
* Resolve "Undefined" rows in meal, market_segment and distribution_channel


---

## Change working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [ ]:
import os
current_dir = os.getcwd()
current_dir

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chdir() defines the new current directory

In [ ]:
os.chdir(os.path.dirname(current_dir))

current_dir = os.getcwd()
current_dir


## Load Data

In [ ]:
import pandas as pd
df_raw = pd.read_csv("outputs/datasets/collection/HotelBookings.csv")
df_raw.head(3)

---

## Outliers

1. Bookings with no guests of any description should be removed. This are impossible bookings and as such are a data validity issue and should be executed fully in the notebook.

In [ ]:
df_raw.shape

In [ ]:
no_guest_df = df_raw[((df_raw["adults"] == 0) & (df_raw["children"] == 0) & (df_raw["babies"] == 0))]
no_guest_df.shape

In [ ]:
df_valid = df_raw.drop(no_guest_df.index)
df_valid.shape

2. Bookings with 5 or more adults should be removed. These are also 'impossible bookings' and should be removed.

In [ ]:
many_adults_df = df_valid[df_valid["adults"] >= 5]
many_adults_df.shape

In [ ]:
df_valid = df_valid.drop(many_adults_df.index)
df_valid.shape

3. Bookings with more than 4 babies or children and bookings with more than 2 car park spaces should be removed. Another case of very rare 'impossible bookings' to be handled in the notebook not the pipeline

In [ ]:
high_values_df = df_valid[((df_valid["children"] > 4) | (df_valid["babies"] > 4) | (df_valid["required_car_parking_spaces"] > 2))]
high_values_df               

In [ ]:
df_valid = df_valid.drop(high_values_df.index)


In [ ]:
df_valid.head()

* Re-run high values check to ensure changes have taken effect

In [ ]:
high_values_df = df_valid[((df_valid["children"] > 4) | (df_valid["babies"] > 4) | (df_valid["required_car_parking_spaces"] > 2))]
high_values_df

4. Remove negative values for `adr`

In [ ]:
df_raw["adr"].describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(18, 6))
sns.boxplot(data=df_raw,
                x="reserved_room_type",
                y="adr",
                ax=ax)

plt.show()

* We can see from the chart that only one value exceeds 1000 for adr, we will handle this outlier in the pipeline, but remove from the clean eda dataset later in this notebook.

In [ ]:
negative_adr = df_valid[(df_valid["adr"] < 0)]
negative_adr

In [ ]:
df_valid = df_valid.drop(negative_adr.index)
df_valid["adr"].describe()

---

## Drop data leakage columns

Since the initial model is intended to run on live enquiry data, it should not be trained on any data not available at time of booking. Four columns here provide data that is not available on live data: `reservation_status`, `reservation_status_date`, `assigned_room_type`, `booking_changes`.

* These 4 variables should be dropped
* `days_in_waiting_list` was considered for dropping also but was retained to allow users to select the best prospect from waitlisted prospects when a room becomes available. For a new enquiry with no wailist history, this value defaults to zero

In [ ]:
df_valid = df_valid.drop(columns=["reservation_status", "reservation_status_date", "assigned_room_type", "booking_changes"])
df_valid.head()

---

## Rows with "Undefined" values

In [ ]:
cols = ["meal", "market_segment", "distribution_channel"]
undefined_rows = df_valid[df_valid[cols].isin(["Undefined"]).any(axis=1)]
undefined_rows.head()

In [ ]:
undefined_rows.shape

* Over the 3 variables, there are 1174 rows to remove totalling less than 1% of the data

In [ ]:
undefined_counts = df_valid[cols].apply(lambda x: (x == "Undefined").sum())
undefined_counts

In [ ]:
undefined_counts.sum()

* The majority of the rows are from the `meal` column and since the total of undefined value counts exceeds the total number of affected rows, we must conclude that some rows have undefined in more than one variable.

In [ ]:
df_valid["meal"].value_counts()

* Since the descriptive notes on the dataset state "Undefined/SC – no meal package", the appropriate method for handling the meal rows is to replace "Undefined" with "SC". This step belongs in the pre-processing pipeline.

In [ ]:
df_clean = df_valid.copy()

df_clean["meal"] = df_clean["meal"].replace("Undefined", "SC")
df_clean["meal"].value_counts()

In [ ]:
undefined_rows = df_clean[df_clean.isin(["Undefined"]).any(axis=1)]
undefined_rows.shape

* These rows can now be dropped from the valid dataset and the fully cleaned dataset

In [ ]:
df_clean = df_clean.drop(undefined_rows.index)
df_valid = df_valid.drop(undefined_rows.index)

In [ ]:
undefined_rows = df_clean[df_clean.isin(["Undefined"]).any(axis=1)]
undefined_rows

In [ ]:
undefined_valid = df_valid[df_valid.isin(["Undefined"]).any(axis=1)]
undefined_valid.shape

---

### High adr value

In [ ]:
df_clean = df_clean.drop(df_valid[((df_valid["adr"] > 1000))].index)
df_clean["adr"].describe()

### Missingness

* Missing values imputation is a pipeline action so these corrections should only be saved to the fully cleaned dataset for analysis only purposes

In [ ]:
# The following function is adapted from the 'Churnometer' walkthrough
def evaluate_missing_data(df):
    missing_data_absolute = df.isnull().sum()
    missing_data_percentage = round(missing_data_absolute/len(df)*100, 2)
    df_missing_data = (pd.DataFrame(
                            data={"RowsWithMissingData": missing_data_absolute,
                                   "PercentageOfDataset": missing_data_percentage,
                                   "DataType": df.dtypes}
                                    )
                          .sort_values(by=['PercentageOfDataset'], ascending=False)
                          .query("RowsWithMissingData > 0")
                          )

    return df_missing_data

In [ ]:
evaluate_missing_data(df_clean)

* As per received guidance [Revenue Manager's Notebook](/jupyter_notebooks/03_rm_analysis.ipynb), replace missing values in `children`, `agent` and `company` with '0'

* The rows with missing children have been removed in a prior cleaning step.

* Check that 0 is not in use for `agent` or `company`

In [ ]:
df_raw[["company", "agent"]].describe()

* The minimum values confirm that 0 is not present in the existing values

In [ ]:
def replace_zeros(df):
    cols_to_replace_zero = ["children", "company", "agent"]
    for col in cols_to_replace_zero:
        df_clean[col] = df_clean[col].fillna(0)
    return df_clean    

replace_zeros(df_clean)

In [ ]:
df_clean.head(10)

In [ ]:
evaluate_missing_data(df_clean)

* For `country` we should replace the missing values with the variable mode value

In [ ]:
df_raw["country"].mode()

* Replace missing values with "PRT" and re-evaluate missingness

In [ ]:
df_clean["country"] = df_clean["country"].fillna("PRT")
evaluate_missing_data(df_clean)

## Type Conversions

1. Children float64 should be int64 since partial children are not possible

In [ ]:
df_clean["children"].dtypes

In [ ]:
df_clean["children"] = df_clean["children"].astype("int64")
df_clean["children"].dtypes

2. Numeric-as-categoric conversion

In [ ]:
df_clean.dtypes

* Convert `is_repeated_guest`, `agent` and `company` to categorical features 

In [ ]:
# First replace float64 dtypes with int64
float_list = ["agent", "company"]
for col in float_list:
    df_clean[col] = df_clean[col].astype("int64")

df_clean.info()

In [ ]:
categorical_list = ["is_repeated_guest", "agent", "company"]
for col in categorical_list:
    df_clean[col] = df_clean[col].astype("category")

df_clean.info()

* `is_canceled` is kept as numeric because the point-biserial correlation in the [correlation study](/jupyter_notebooks/05_correlation_study.ipynb) requires a numeric binary target
* `arrival_date_year` and `arrival_date_day_of_month` are kept as numeric for datetime amalgamation in [feature engineering](/jupyter_notebooks/07_feature_engineering.ipynb)
* `arrival_date_week_number` is kept as numeric due to its cyclical nature, a full categorisation can be performed in [feature engineering](/jupyter_notebooks/07_feature_engineering.ipynb)

## Save Files

* Save the cleaned dataframe for future use

In [ ]:
try:
  os.makedirs(name='outputs/datasets/cleaned')
except Exception as e:
  print(e)


In [ ]:
df_clean.to_csv("outputs/datasets/cleaned/HotelBookingsClean.csv", index=False)
df_valid.to_csv("outputs/datasets/cleaned/HotelBookingsValid.csv", index=False)

---

## Conclusions

* Compare cleaned dataset to original

In [ ]:
df_raw = pd.read_csv("outputs/datasets/collection/HotelBookings.csv")
df_raw.info()

In [ ]:
print(f"Original dataset had shape: {df_raw.shape}.")
print(f"Valid dataset has shape: {df_valid.shape}.")
print(f"Clean dataset has shape: {df_clean.shape}.")

* Original dataset (119,390 rows) was processed into two outputs:
    * HotelBookingsValid.csv – a data-valid dataset retaining missing values and documented placeholder values for preprocessing within the modelling pipeline.
    * HotelBookingsClean.csv – a fully cleaned dataset for exploratory analysis, statistical testing and feature exploration.
* Removed invalid records that could not represent genuine bookings, including bookings with no guests, implausible guest counts, excessive children/babies/car parking requests and negative ADR values.
* The validated dataset has shape (119180, 28) and the fully cleaned dataset has shape (119179), saved to outputs/datasets/cleaned/HotelBookingsValid.csv and outputs/datasets/cleaned/HotelBookingsValid.csv

**Next Steps**
* The cleaned dataset is ready for the [Correlation Study](/jupyter_notebooks/05_correlation_study.ipynb), which will assess relationships between features and `is_canceled`
* The valid dataset is ready for the preprocessing pipeling in [feature engineering](/jupyter_notebooks/07_feature_engineering.ipynb)